# AML Signal Escalation — Submission Pipeline

Track A (feature engineering) va Track B (modeling) qismlarini boshidan oxirigacha birlashtiruvchi reproducible notebook.
Ishga tushirishdan oldin loyiha ildizidan `pip install -r requirements.txt` bajaring.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))

from src import config
from src.data_loading import load_signals, load_transactions
from src.features import build
from src.model import train, cross_validate
from src.predict import predict, validate_submission, write

## 1. Load raw data

In [ ]:
train_signals = load_signals(config.TRAIN_SIGNALS_PATH)
train_transactions = load_transactions(config.TRAIN_TRANSACTIONS_PATH)
test_signals = load_signals(config.TEST_SIGNALS_PATH)
test_transactions = load_transactions(config.TEST_TRANSACTIONS_PATH)
train_signals.shape, train_transactions.shape, test_signals.shape, test_transactions.shape

## 2. Build features (Track A)

In [ ]:
train_features = build(train_signals, train_transactions)
test_features = build(test_signals, test_transactions)
train_features.head()

## 3. Train + cross-validate (Track B)

In [ ]:
X = train_features[config.FEATURE_COLUMNS]
y = train_signals[config.TARGET_COL]

auc = cross_validate(X, y)
print(f"CV ROC-AUC: {auc:.4f}")

model = train(X, y)

## 4. Predict + write submission

In [ ]:
predictions = predict(model, test_features)
validate_submission(predictions, test_signals[config.ID_COL])
write(predictions, str(config.OUTPUTS_DIR / "team_<TEAM_ID>.csv"))
predictions.head()